# elutediff · 01 · Data & RT-density targets

Runs on a **free CPU Colab runtime**. This notebook loads METLIN SMRT (or a synthetic stand-in), audits the 256-token canvas budget, builds the Gaussian RT-density targets, and writes the `(prompt, target)` JSONL that the baseline and DiffusionGemma notebooks consume.

See `docs/density-first-revision.md` for the science and `docs/project-structure.md` for the code map.

In [ ]:
# Install elutediff from GitHub (core + chemistry + baselines extras).
%pip install -q "elutediff[chem,baselines] @ git+https://github.com/CodeHalwell/elutediff.git@claude/project-structure-unsloth-bnwe57"
%pip install -q matplotlib

## 1. Get the data

**Option A (real):** download METLIN SMRT (figshare DOI `10.6084/m9.figshare.8038913`) and set `METLIN_PATH` to the `.sdf`/`.csv`.

**Option B (demo):** the cell below writes a small synthetic SDF so the notebook is runnable end-to-end out of the box.

In [ ]:
# --- Option B: synthetic demo data so this notebook runs without the download ---
# Replace METLIN_PATH with your real SDF/CSV to use the actual dataset.
import random
from rdkit import Chem
from rdkit.Chem import Descriptors

def write_demo_sdf(path, n=400, seed=0):
    rng = random.Random(seed)
    cores = ['c1ccccc1', 'c1ccncc1', 'C1CCCCC1', 'c1ccc2ccccc2c1', 'C1CCNCC1']
    w = Chem.SDWriter(path)
    written = 0
    for i in range(n):
        core = rng.choice(cores)
        chain = 'C' * rng.randint(0, 8)
        tail = rng.choice(['', 'O', 'N', 'C(=O)O', 'Cl', 'OC'])
        smi = chain + core + tail
        m = Chem.MolFromSmiles(smi)
        if m is None:
            continue
        # A retention-relevant synthetic RT: driven by LogP + MolWt + noise.
        rt = 60 + 8.0 * Descriptors.MolLogP(m) * 10 + 0.8 * Descriptors.MolWt(m) + rng.uniform(-20, 20)
        rt = max(20.0, min(rt, 1180.0))
        m.SetProp('_Name', f'CID{i}')
        m.SetProp('RETENTION_TIME', f'{rt:.1f}')
        w.write(m)
        written += 1
    w.close()
    return written

METLIN_PATH = '/content/demo_smrt.sdf'
print('demo molecules:', write_demo_sdf(METLIN_PATH))

In [ ]:
from elutediff.data.metlin import load_metlin

molecules, stats = load_metlin(METLIN_PATH, return_stats=True)
print(stats)
molecules[:3]

## 2. Configure the target & audit the canvas

Bin width and sigma are coupled: the Gaussian should span ~2-3 bins so the density is resolved rather than a spike. The audit is **mandatory** before training — the target must fit the 256-token canvas.

In [ ]:
from elutediff.config import Config
from elutediff.audit import sweep_bin_widths

cfg = Config()
cfg.metlin_path = METLIN_PATH
cfg.target.rt_max = 1200.0   # set from your RT 99-99.5th percentile
cfg.target.bin_width = 10.0
cfg.target.sigma = 20.0

for res in sweep_bin_widths(cfg.target, bin_widths=(10.0, 5.0),
                            canvas_length=cfg.model.canvas_length):
    print(('OK  ' if res.fits_canvas else 'OVER'), res.summary())

## 3. Visualize one RT-density target

Scalar RT &rarr; Gaussian density &rarr; max-normalize &rarr; quantize to `000..100` fixed-width tokens.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from elutediff.targets.density import gaussian_density, time_grid
from elutediff.targets.quantize import quantize
from elutediff.serialization.prompts import format_rt_vector

rt = molecules[0].rt_seconds
grid = time_grid(cfg.target)
dens = gaussian_density(rt, cfg.target)
levels = quantize(dens, cfg.target)

plt.figure(figsize=(9, 3))
plt.plot(grid, levels, marker='.')
plt.axvline(rt, color='r', ls='--', label=f'true RT = {rt:.0f}s')
plt.xlabel('retention time (s)'); plt.ylabel('quantized intensity'); plt.legend()
plt.title('RT-density target'); plt.show()

print(format_rt_vector(levels, cfg.target)[:300], '...')

## 4. Build the (prompt, target) JSONL + splits

Applies the configured split (scaffold by default) and writes rows tagged with their fold. Conditioning level 1 = SMILES only; bump `cfg.conditioning.level` to add descriptors / atom-bond tables.

In [ ]:
import json
from elutediff.data.splits import make_split
from elutediff.targets.noise import apply_noise
from elutediff.serialization.prompts import build_prompt, target_string
from elutediff.data.molecules import compute_descriptors

# Scaffold split is the headline for real METLIN data; the synthetic demo has
# only a few ring systems, so use a random split here to keep folds populated.
cfg.split.strategy = 'random'
smiles = [m.smiles for m in molecules]
split = make_split(smiles, cfg.split)
fold = {i: name for name, idxs in
        (('train', split.train_idx), ('val', split.val_idx), ('test', split.test_idx))
        for i in idxs}
print('train/val/test:', split.sizes())

OUT = '/content/targets.jsonl'
cond = cfg.conditioning
with open(OUT, 'w') as fh:
    for i, m in enumerate(molecules):
        levels = quantize(apply_noise(gaussian_density(m.rt_seconds, cfg.target), cfg.noise), cfg.target)
        desc = compute_descriptors(m.smiles, cond.descriptors) if cond.level >= 2 else None
        fh.write(json.dumps({
            'smiles': m.smiles, 'rt': m.rt_seconds, 'pubchem': m.pubchem_id,
            'split': fold.get(i, 'train'),
            'prompt': build_prompt(smiles=m.smiles, target_cfg=cfg.target, cond_cfg=cond, descriptors=desc),
            'target': target_string(levels, cfg.target),
        }) + '\n')
print('wrote', OUT)
print(open(OUT).readline()[:300])

**Next:** open `02_baselines.ipynb` (CPU) for the known-bar regressors, then `03_diffusiongemma_finetune.ipynb` (A100) for the diffusion model. To carry the JSONL across notebooks, save it to Google Drive or re-run this notebook.